# 弹簧振子引力辐射计算

本 Notebook 实现引力波物理中的基础流程，以**弹簧 + 两端质点**为例：

1. **系统描述** — 弹簧原长 $l_0$，劲度系数 $k$，两端各连质量 $m$ 的质点，
   从压缩长度 $l$ 释放。
2. **四极矩** — $I_{ij}(t)$ 及其无迹化（reduced）形式 $Q_{ij}(t)$
3. **TT 规范投影** — 横向无迹（TT-gauge）投影算符 $\Lambda_{ijkl}$
4. **引力辐射** — 度规扰动 $h_{ij}^{\mathrm{TT}}(t)$

$$h_{ij}^{\mathrm{TT}} = \frac{2G}{c^4 r}\,\Lambda_{ijkl}\,\ddot{Q}_{kl}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# 绘图中文字体回退（Colab/本地均适用）
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

## 1  系统参数

两质点沿 $z$ 轴排列，质心固定在原点。
质心坐标系中质点位置为 $z_1 = -r/2$，$z_2 = +r/2$，$r$ 为弹簧当前长度。

In [ ]:
# ── 物理参数 ────────────────────────────────────────────────────────────────
m  = 1.0     # kg  各质点质量
k  = 10.0    # N/m 弹簧劲度系数
l0 = 1.0     # m   弹簧原长
l  = 0.6     # m   初始压缩长度

G_N = 6.674e-11  # N m² kg⁻²  引力常数
c   = 3e8        # m/s  光速

# ── 导出量 ──────────────────────────────────────────────────────────────────
omega = np.sqrt(2*k/m)   # 相对振动角频率
A     = l - l0           # 振幅（压缩：负值）
T_osc = 2*np.pi / omega  # 振动周期

print(f'角频率  ω = {omega:.4f} rad/s')
print(f'振动周期 T = {T_osc:.4f} s')
print(f'振幅    A = {A:.3f} m  (初始压缩 {-A/l0*100:.0f}%)')

## 2  运动方程与解析解

质心坐标守恒，只需追踪相对坐标 $r = z_2 - z_1$：

$$\ddot r = -\frac{2k}{m}(r - l_0) \implies r(t) = l_0 + A\cos(\omega t)$$

其中 $\omega = \sqrt{2k/m}$，$A = l - l_0$。

In [ ]:
# ── 数值验证：通过 ODE 求解器求解 ─────────────────────────────────────────
def eom(t, y):
    """y = [r, r_dot]"""
    r, rdot = y
    rddot = -(2*k/m) * (r - l0)
    return [rdot, rddot]

t_span = (0, 5*T_osc)
t_eval = np.linspace(0, 5*T_osc, 5000)
sol = solve_ivp(eom, t_span, [l, 0], t_eval=t_eval, rtol=1e-10)

r_num  = sol.y[0]
r_anal = l0 + A * np.cos(omega * t_eval)

# ── 解析解 ────────────────────────────────────────────────────────────────
r      = r_anal
r_dot  = -A * omega * np.sin(omega * t_eval)
r_ddot = -A * omega**2 * np.cos(omega * t_eval)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_eval/T_osc, r_anal, label='解析解', lw=2)
ax.plot(t_eval/T_osc, r_num,  label='数值解 (ODE)', lw=1.5, ls='--', alpha=0.7)
ax.axhline(l0, color='grey', ls=':', label=f'原长 $l_0={l0}$ m')
ax.set_xlabel('时间 $t/T$')
ax.set_ylabel('弹簧长度 $r$ (m)')
ax.set_title('弹簧振子：相对坐标 $r(t)$')
ax.legend()
plt.tight_layout()
plt.show()
print(f'数值 vs 解析最大误差: {np.max(np.abs(r_num - r_anal)):.2e} m')

## 3  四极矩

### 3.1  质量四极矩 $I_{ij}$

$$I_{ij} = \sum_a m_a\, x_a^i\, x_a^j$$

两质点位于 $(0,0,\pm r/2)$，所以只有 $I_{zz}$ 非零：

$$I_{zz} = m\left(\frac{r}{2}\right)^2 + m\left(-\frac{r}{2}\right)^2 = \frac{mr^2}{2}$$

In [ ]:
t_arr = t_eval
z1 = -r/2;  z2 = r/2   # 质心坐标系位置

I_xx = np.zeros_like(t_arr)
I_yy = np.zeros_like(t_arr)
I_zz = m * z1**2 + m * z2**2   # = m r²/2
I_trace = I_xx + I_yy + I_zz

# ── 以矩阵形式展示某一时刻的 I_ij ─────────────────────────────────────────
idx = 0
I_mat = np.diag([I_xx[idx], I_yy[idx], I_zz[idx]])
print('t=0 时刻的质量四极矩张量 I_ij (kg·m²):')
print(I_mat)

### 3.2  无迹（reduced / STF）四极矩 $Q_{ij}$

$$Q_{ij} = I_{ij} - \frac{1}{3}\delta_{ij}\,I_{kk}$$

In [ ]:
Q_xx = I_xx - I_trace/3
Q_yy = I_yy - I_trace/3
Q_zz = I_zz - I_trace/3

print('验证无迹条件 Q_xx+Q_yy+Q_zz =', np.max(np.abs(Q_xx+Q_yy+Q_zz)), '(应为 0)')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(t_arr/T_osc, I_zz, label=r'$I_{zz}$')
axes[0].plot(t_arr/T_osc, I_xx, label=r'$I_{xx}=I_{yy}$', ls='--')
axes[0].set_xlabel('$t/T$');  axes[0].set_ylabel('$I_{ij}$ (kg·m²)')
axes[0].set_title('质量四极矩 $I_{ij}(t)$');  axes[0].legend()

axes[1].plot(t_arr/T_osc, Q_zz, label=r'$Q_{zz}$', color='C2')
axes[1].plot(t_arr/T_osc, Q_xx, label=r'$Q_{xx}=Q_{yy}$', ls='--', color='C3')
axes[1].set_xlabel('$t/T$');  axes[1].set_ylabel('$Q_{ij}$ (kg·m²)')
axes[1].set_title('无迹四极矩 $Q_{ij}(t)$');  axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3  四极矩的二阶时间导数 $\ddot{Q}_{ij}$

解析表达式（$r = l_0 + A\cos\omega t$）：

$$\ddot{I}_{zz} = m\bigl(\dot r^2 + r\ddot r\bigr), \quad\ddot Q_{zz} = \tfrac{2}{3}\ddot I_{zz}, \quad\ddot Q_{xx} = \ddot Q_{yy} = -\tfrac{1}{3}\ddot I_{zz}$$

In [ ]:
I_zz_ddot_anal = m * (r_dot**2 + r * r_ddot)

Q_zz_ddot = (2/3) * I_zz_ddot_anal
Q_xx_ddot = -(1/3) * I_zz_ddot_anal
Q_yy_ddot = -(1/3) * I_zz_ddot_anal

# 数值微分验证
dt = t_arr[1] - t_arr[0]
Q_zz_ddot_num = np.gradient(np.gradient(Q_zz, dt), dt)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_arr/T_osc, Q_zz_ddot, label=r'解析 $\ddot Q_{zz}$', lw=2)
ax.plot(t_arr/T_osc, Q_zz_ddot_num, label=r'数值微分 $\ddot Q_{zz}$',
        ls='--', alpha=0.7)
ax.set_xlabel('$t/T$');  ax.set_ylabel(r'$\ddot Q_{zz}$ (kg·m²/s²)')
ax.set_title(r'四极矩二阶时间导数 $\ddot Q_{zz}(t)$');  ax.legend()
plt.tight_layout()
plt.show()

## 4  TT 规范：投影算符

对于沿方向 $\hat n$ 传播的引力波，**横向无迹（TT）投影算符**为

$$\Lambda_{ijkl} = P_{ik}P_{jl} - \tfrac{1}{2}P_{ij}P_{kl}$$

其中横向投影子 $P_{ij} = \delta_{ij} - n_i n_j$。

作用在对称无迹张量 $Q_{kl}$ 上：

$$\bigl(\Lambda \cdot Q\bigr)_{ij} = (P\,Q\,P)_{ij} - \tfrac{1}{2}P_{ij}\operatorname{Tr}(P\,Q)$$

In [ ]:
def tt_project(Q_arr, n):
    """
    将 TT 投影算符作用于对称张量 Q（时间序列）。

    参数
    ----
    Q_arr : array, shape (N, 3, 3)  随时间变化的张量
    n     : array, shape (3,)       传播方向单位向量

    返回
    ----
    h_TT : array, shape (N, 3, 3)
    """
    I3 = np.eye(3)
    P  = I3 - np.outer(n, n)           # 横向投影子
    # (Λ Q)_ij = (P Q P)_ij - 0.5 P_ij Tr(P Q)
    PQP    = np.einsum('ik,Nkl,lj->Nij', P, Q_arr, P)
    trPQ   = np.einsum('ij,Nji->N', P, Q_arr)    # Tr(PQ) for each time step
    h_TT   = PQP - 0.5 * trPQ[:, None, None] * P[None, :, :]
    return h_TT


# 展示某时刻的投影算符（以矩阵形式）
def show_projector(n, label):
    I3 = np.eye(3)
    P = I3 - np.outer(n, n)
    print(f'传播方向 {label}  的横向投影子 P_ij =')
    print(P.astype(int))
    print()

for n_hat, lbl in [([1,0,0],'x̂'), ([0,1,0],'ŷ'), ([0,0,1],'ẑ')]:
    show_projector(np.array(n_hat, float), lbl)

## 5  引力辐射 $h_{ij}^{\mathrm{TT}}$

远场度规扰动：

$$h_{ij}^{\mathrm{TT}} = \frac{2G}{c^4 r}\,\Lambda_{ijkl}\,\ddot{Q}_{kl}$$

本例取观测点距离 $r = 1\,$pc（秒差距），观测方向为 $\hat n = \hat x$（赤道方向）。

> **注**：对于沿 $z$ 轴振动的系统，沿 $z$ 方向（极轴）辐射为零；
> 沿 $x$ 或 $y$ 方向辐射最强。

In [ ]:
r_obs = 3.086e16   # 1 pc (m)

# 构造 Q̈_ij 的时间序列（3×3 矩阵）
N = len(t_arr)
Q_ddot_arr = np.zeros((N, 3, 3))
Q_ddot_arr[:, 0, 0] = Q_xx_ddot   # xx
Q_ddot_arr[:, 1, 1] = Q_yy_ddot   # yy
Q_ddot_arr[:, 2, 2] = Q_zz_ddot   # zz

# TT 投影（三个方向）
n_x = np.array([1, 0, 0])
n_y = np.array([0, 1, 0])
n_z = np.array([0, 0, 1])

prefactor = 2 * G_N / (c**4 * r_obs)

h_x = prefactor * tt_project(Q_ddot_arr, n_x)   # 观测沿 x̂
h_y = prefactor * tt_project(Q_ddot_arr, n_y)   # 观测沿 ŷ
h_z = prefactor * tt_project(Q_ddot_arr, n_z)   # 观测沿 ẑ

print('验证 TT 条件（沿 x̂ 方向）：')
print(f'  迹 Tr(h_TT) max = {np.max(np.abs(h_x[:,0,0]+h_x[:,1,1]+h_x[:,2,2])):.2e}  (应为 0)')
print(f'  横向 n_i h_ij max = {np.max(np.abs(h_x[:,0,:])):.2e}  (应为 0)')

In [ ]:
# 极化分量 h_+ 和 h_×
# 对于沿 x̂ 传播，TT 平面为 y-z 平面
#   h_+  = (h_yy - h_zz)/2
#   h_×  = h_yz

h_plus_x  = (h_x[:, 1, 1] - h_x[:, 2, 2]) / 2
h_cross_x = h_x[:, 1, 2]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t_arr/T_osc, h_plus_x, color='steelblue', lw=1.5)
axes[0].set_ylabel(r'$h_+$')
axes[0].set_title(r'引力波极化分量（观测方向 $\hat n = \hat x$，$r=1\,$pc）')
axes[0].axhline(0, color='k', lw=0.5)

axes[1].plot(t_arr/T_osc, h_cross_x, color='coral', lw=1.5)
axes[1].set_ylabel(r'$h_\times$')
axes[1].set_xlabel(r'时间 $t/T$')
axes[1].axhline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()
print(f'h_+ 振幅 ≈ {np.max(np.abs(h_plus_x)):.3e}')
print(f'h_× 振幅 ≈ {np.max(np.abs(h_cross_x)):.3e}')

## 6  方向辐射图样（Angular Pattern）

在单次时刻扫描全天球，计算 $h_+$ 振幅随方向 $\hat n(\theta,\varphi)$ 的分布。

In [ ]:
# 对每个方向计算 |h_+| 振幅（取时间最大值）
Ntheta, Nphi = 60, 120
theta_grid = np.linspace(0, np.pi, Ntheta)
phi_grid   = np.linspace(0, 2*np.pi, Nphi)
TH, PH = np.meshgrid(theta_grid, phi_grid, indexing='ij')

H_amp = np.zeros((Ntheta, Nphi))
for it in range(Ntheta):
    for ip in range(Nphi):
        th_val = theta_grid[it]
        ph_val = phi_grid[ip]
        nn = np.array([np.sin(th_val)*np.cos(ph_val),
                       np.sin(th_val)*np.sin(ph_val),
                       np.cos(th_val)])
        h_tmp = prefactor * tt_project(Q_ddot_arr, nn)
        # h_+ 在该方向的最大振幅（简化：取 Frobenius 范数）
        H_amp[it, ip] = np.max(np.sqrt(np.sum(h_tmp**2, axis=(1,2))))

# 转为笛卡尔坐标用于 3D 可视化
R_plot = H_amp / H_amp.max()   # 归一化
X = R_plot * np.sin(TH) * np.cos(PH)
Y = R_plot * np.sin(TH) * np.sin(PH)
Z = R_plot * np.cos(TH)

fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, facecolors=plt.cm.viridis(R_plot), alpha=0.85)
ax.set_xlabel('x');  ax.set_ylabel('y');  ax.set_zlabel('z')
ax.set_title('引力波辐射强度方向图（归一化）\n沿 z 轴方向辐射为零')
plt.tight_layout()
plt.show()

## 7  $h_{ij}^{\mathrm{TT}}$ 张量的矩阵可视化

以热图（heatmap）方式展示多个时刻的 $h_{ij}^{\mathrm{TT}}$ 矩阵。

In [ ]:
n_snapshots = 6
snap_indices = np.linspace(0, N-1, n_snapshots, dtype=int)

fig, axes = plt.subplots(1, n_snapshots, figsize=(14, 2.5))
vmax = np.max(np.abs(h_x))
labels = ['xx','xy','xz','yx','yy','yz','zx','zy','zz']

for i, idx in enumerate(snap_indices):
    ax = axes[i]
    im = ax.imshow(h_x[idx], cmap='RdBu', vmin=-vmax, vmax=vmax,
                   interpolation='nearest', aspect='equal')
    ax.set_xticks([0,1,2]);  ax.set_xticklabels(['x','y','z'], fontsize=8)
    ax.set_yticks([0,1,2]);  ax.set_yticklabels(['x','y','z'], fontsize=8)
    ax.set_title(f'$t = {t_arr[idx]/T_osc:.2f}T$', fontsize=9)

plt.suptitle(r'$h_{ij}^{\rm TT}(t)$ 矩阵（观测方向 $\hat n=\hat x$）', y=1.04)
plt.colorbar(im, ax=axes, shrink=0.8, label=r'$h_{ij}$')
plt.tight_layout()
plt.show()

## 8  小结与展望

本 Notebook 完整展示了从力学系统到引力辐射的基础流程：

| 步骤 | 物理量 | 计算方法 |
|------|--------|----------|
| 1 | 运动方程 | ODE 数值积分 / 解析解 |
| 2 | 质量四极矩 $I_{ij}$ | 质量分布求和 |
| 3 | 无迹四极矩 $Q_{ij}$ | $Q=I-\frac{1}{3}\delta\,\mathrm{Tr}(I)$ |
| 4 | TT 投影算符 $\Lambda_{ijkl}$ | $P_{ik}P_{jl}-\frac{1}{2}P_{ij}P_{kl}$ |
| 5 | 引力波 $h_{ij}^{\rm TT}$ | $\frac{2G}{c^4 r}\Lambda_{ijkl}\ddot Q_{kl}$ |

**下一步扩展**：
- 双星系统（Kepler 轨道）的四极矩与 GW
- 椭圆轨道的 $h_+$, $h_\times$ 时间序列
- 匹配滤波与引力波探测器响应
